# Station Dimension Data - Gold Layer

## Objective
Extract and deduplicate unique station attribute data from silver.silver_velib to build a standardized Station Dimension Gold Delta table (gold.gold_dim_station).

## Data Flow
silver.silver_velib → Spark SQL / DataFrame → gold.gold_dim_station

## Source
The underlying data comes from the Paris OpenData API: Vélib' - Emplacements des stations.

## Input
Silver Delta table: silver.silver_velib

## Output
Gold Delta table: gold.gold_dim_station

## Gold Layer Principle
The Gold layer delivers curated, dimensional models and business-level aggregations ready for reporting and analytics. This pipeline isolates unique physical stations, spatial coordinates, and station metadata to maintain a clean dimension table with strict entity integrity.

## Processing Steps
1. **Load Silver Data:** Read silver.silver_velib into PySpark and display initial dataset.
2. **Extract Dimension:** Query distinct station codes, station names, coordinates, and capacity.
3. **Write to Gold:** Persist deduplicated dataset to gold.gold_dim_station Delta table.

In [0]:
# Load data from silver schema
df_silver_velib=spark.table('workspace.silver.silver_velib')

In [0]:
# display the dataframe 
df_silver_velib.display()

## BUSINESS TRANSFORMATION AND MODELING

In [0]:
# Extract Station Dimension Data
query_dim_station = """
SELECT DISTINCT
    stationcode AS station_code,
    name AS station_name,
    capacity,
    CAST(coordonnees_geo.lat AS DOUBLE) AS latitude,
    CAST(coordonnees_geo.lon AS DOUBLE) AS longitude,
    nom_arrondissement_communes AS commune_name,
    code_insee_commune AS code_insee
FROM silver.silver_velib
"""

df_dim_station = spark.sql(query_dim_station)

In [0]:
# Display df_dim_station 
df_dim_station.display()

# WRITING GOLD TABLE

In [0]:
# writing df_dim_station to gold schema
df_dim_station\
    .write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true") \
        .saveAsTable("gold.gold_dim_station")


# CHECKING THE GOLD TABLE

In [0]:
%sql
SELECT * 
FROM workspace.gold.gold_dim_stationworkspace.gold.gold_fact_status;workspace.gold.gold_fact_status